<a href="https://colab.research.google.com/github/Rajfekar/PythonML/blob/main/Raj_Copy_VAE_CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# Install (usually preinstalled in Colab)
# ==========================================
!pip install torch torchvision --quiet

# ==========================================
# Imports
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
# ==========================================
# CIFAR-10 Dataset
# 32x32 RGB images (10 classes)
# ==========================================

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

print("Training samples:", len(train_dataset))

100%|██████████| 170M/170M [00:03<00:00, 44.9MB/s]


Training samples: 50000


In [3]:
# ==========================================
# VAE Model
# ==========================================

class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(VAE, self).__init__()

        # ======================
        # Encoder (CNN)
        # ======================
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),  # 32x16x16
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 64x8x8
            nn.ReLU(),
            nn.Flatten()
        )

        self.fc_mu = nn.Linear(64*8*8, latent_dim)
        self.fc_logvar = nn.Linear(64*8*8, latent_dim)

        # ======================
        # Decoder
        # ======================
        self.fc_decode = nn.Linear(latent_dim, 64*8*8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), # 32x16x16
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),  # 3x32x32
            nn.Sigmoid()  # Output in [0,1]
        )

    def encode(self, x):
        x = self.encoder(x)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    # Reparameterization Trick
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        return mu + epsilon * std

    def decode(self, z):
        x = self.fc_decode(z)
        x = x.view(-1, 64, 8, 8)
        return self.decoder(x)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


latent_dim = 128
model = VAE(latent_dim).to(device)

In [4]:
# ==========================================
# VAE Loss
# Reconstruction Loss + KL Divergence
# ==========================================

def vae_loss(recon_x, x, mu, logvar):

    # Reconstruction loss (MSE)
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')

    # KL Divergence
    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return recon_loss + kl_loss

In [5]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 50
loss_history = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, _ in train_loader:
        images = images.to(device)

        optimizer.zero_grad()

        recon, mu, logvar = model(images)
        loss = vae_loss(recon, images, mu, logvar)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataset)
    loss_history.append(avg_loss)

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

Epoch [1/50] Loss: 109.6095
Epoch [2/50] Loss: 81.9762
Epoch [3/50] Loss: 79.3686
Epoch [4/50] Loss: 78.4831
Epoch [5/50] Loss: 77.9371
Epoch [6/50] Loss: 77.3342
Epoch [7/50] Loss: 76.9155
Epoch [8/50] Loss: 76.5900
Epoch [9/50] Loss: 76.2695
Epoch [10/50] Loss: 76.0797
Epoch [11/50] Loss: 75.9296
Epoch [12/50] Loss: 75.7037
Epoch [13/50] Loss: 75.5909
Epoch [14/50] Loss: 75.4681
Epoch [15/50] Loss: 75.4185
Epoch [16/50] Loss: 75.2807
Epoch [17/50] Loss: 75.2522
Epoch [18/50] Loss: 75.1278
Epoch [19/50] Loss: 75.0870
Epoch [20/50] Loss: 75.0589
Epoch [21/50] Loss: 74.9993
Epoch [22/50] Loss: 74.8872
Epoch [23/50] Loss: 74.8783
Epoch [24/50] Loss: 74.7815
Epoch [25/50] Loss: 74.7853
Epoch [26/50] Loss: 74.7528
Epoch [27/50] Loss: 74.6889
Epoch [28/50] Loss: 74.6603
Epoch [29/50] Loss: 74.6328
Epoch [30/50] Loss: 74.6109
Epoch [31/50] Loss: 74.5808
Epoch [32/50] Loss: 74.5462
Epoch [33/50] Loss: 74.5330
Epoch [34/50] Loss: 74.4801
Epoch [35/50] Loss: 74.4501
Epoch [36/50] Loss: 74.4495


In [ ]:
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    images, _ = next(iter(train_loader))
    images = images[:8].to(device)

    recon, _, _ = model(images)

images = images.cpu()
recon = recon.cpu()

fig, axes = plt.subplots(2, 8, figsize=(15,4))

for i in range(8):
    axes[0, i].imshow(np.transpose(images[i], (1,2,0)))
    axes[0, i].axis("off")

    axes[1, i].imshow(np.transpose(recon[i], (1,2,0)))
    axes[1, i].axis("off")

axes[0,0].set_ylabel("Original")
axes[1,0].set_ylabel("Reconstructed")

plt.show()

In [ ]:
with torch.no_grad():
    z = torch.randn(8, latent_dim).to(device)
    generated = model.decode(z).cpu()

fig, axes = plt.subplots(1, 8, figsize=(15,3))

for i in range(8):
    axes[i].imshow(np.transpose(generated[i], (1,2,0)))
    axes[i].axis("off")

plt.show()

In [ ]:
# what done

# Encoder compresses image to latent vector (μ, logvar)

# Reparameterization trick allows gradient flow

# Decoder reconstructs image

# Loss = Reconstruction + KL divergence

# Latent space sampling allows new image generation

# CIFAR-10 is harder than MNIST due to RGB complexity